In [173]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    f1_score
)

from imblearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')


In [174]:
# Load Data
df = pd.read_csv('../data/Processed_HR_Employee_Attrition.csv')
joblib.dump(df, '../df_original.pkl')

['../df_original.pkl']

In [175]:
# Encode
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df = pd.get_dummies(df, drop_first=True)

# Split
X = df.drop(columns='Attrition')
y = df['Attrition']

In [176]:
# ── Split 1: Train (70%) / Test (30%) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

In [177]:
# ── Split 2: Train_full (80%) / Validation (20%) ──
X_train_full, X_val, y_train_full, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,   # 0.3 * 0.7 = 21% overall
    stratify=y_train,
    random_state=42
)

In [178]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        solver='liblinear',
        max_iter=2000,
        random_state=42
    ))
])

In [179]:
# GridSearch (ONLY on Train_full)
param_grid = {
    'model__C': [0.01, 0.1, 0.2, 0.35, 0.4, 0.45, 0.5, 1, 5]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train_full, y_train_full)

best_model = grid.best_estimator_

print(f"Best C: {grid.best_params_['model__C']}")

/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/utkarshuday/Desktop/employee-attrition/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Best C: 0.4


In [180]:
# ── Validation Phase (Threshold tuning) ──
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

thresholds = np.linspace(0.1, 0.9, 100)

fold_thresholds = []
fold_f1s = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    X_tr, X_val_fold = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_tr, y_val_fold = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

    # Train model on fold
    best_model.fit(X_tr, y_tr)

    # Predict probabilities
    y_val_proba = best_model.predict_proba(X_val_fold)[:, 1]

    best_thresh = 0
    best_f1 = 0

    # Find best threshold for this fold
    for t in thresholds:
        preds = (y_val_proba >= t).astype(int)
        f1 = f1_score(y_val_fold, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    fold_thresholds.append(best_thresh)
    fold_f1s.append(best_f1)

    print(f"Fold {fold+1} → Best Threshold: {best_thresh:.3f}, F1: {best_f1:.4f}")

# Final threshold
final_threshold = np.mean(fold_thresholds)

print("\n── Cross-Validated Threshold ──")
print(f"Thresholds: {fold_thresholds}")
print(f"Final Threshold (mean): {final_threshold:.3f}")
print(f"Mean CV F1: {np.mean(fold_f1s):.4f}")

Fold 1 → Best Threshold: 0.666, F1: 0.6780
Fold 2 → Best Threshold: 0.852, F1: 0.6977
Fold 3 → Best Threshold: 0.779, F1: 0.5833
Fold 4 → Best Threshold: 0.722, F1: 0.4615
Fold 5 → Best Threshold: 0.649, F1: 0.6032

── Cross-Validated Threshold ──
Thresholds: [np.float64(0.6656565656565656), np.float64(0.8515151515151514), np.float64(0.7787878787878788), np.float64(0.7222222222222222), np.float64(0.6494949494949495)]
Final Threshold (mean): 0.734
Mean CV F1: 0.6047


In [181]:
final_model = best_model
final_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(C=0.4, class_weight='balanced',
                                    max_iter=2000, random_state=42,
                                    solver='liblinear'))])

In [189]:
# ── Test Evaluation (FINAL) ──
y_test_proba = final_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.62).astype(int)

print("\n── FINAL TEST EVALUATION ──")
print(classification_report(y_test, y_test_pred))

print(f"ROC-AUC: {roc_auc_score(y_test, y_test_proba):.4f}")


── FINAL TEST EVALUATION ──
              precision    recall  f1-score   support

           0       0.93      0.88      0.91       370
           1       0.52      0.68      0.59        71

    accuracy                           0.85       441
   macro avg       0.73      0.78      0.75       441
weighted avg       0.87      0.85      0.85       441

ROC-AUC: 0.8327
